# 🚀 Building AI Agents with ADK: From Single Agents to Multi-Agent Systems 🚀

Welcome, Agent Architect! This codelab takes you from zero to building sophisticated multi-agent systems using the Google Agent Development Kit (ADK).

### By the end of this adventure, you will be able to:
* **Build a Foundational Agent:** Create an effective AI agent from scratch using the Google Agent Development Kit (ADK).
* **Design Custom Tools:** Connect agents to external APIs and data sources.
* **Orchestrate Multi-Agent Teams:** Build systems where agents collaborate and delegate tasks.
* **Implement Conversational Memory:** Enable agents to maintain context across multi-turn conversations.
* **Apply Advanced Patterns:** Use agent-as-a-tool and hierarchical agent architectures.

**Author:** `[Your Name]` | Passionate about helping developers build intelligent AI systems.

```text
  (_/)
  (•ㅅ•)
  /づ  🏆      Let's build something amazing!
```

---
## 🛑 Important Prerequisite: Setup Your Environment!

Before diving in, ensure you have:
1. A Google Cloud Project with billing enabled.
2. Vertex AI API enabled.
3. Python 3.9+ installed.

```text
   /\/\     /\/\     /\/\      /\/\       /\/\
  ( ^^ )   ( -.- )   ( >< )   ( =^.^= )    ( oo )   
```

---
## Part 0: Setup & Authentication 🔑

First, let's install dependencies and configure authentication.

In [ ]:
!pip install google-adk google-generativeai requests -q

In [ ]:
import os
import sys
import json
import asyncio
import requests
from typing import Any, Dict, List
from google.colab import auth
from IPython.display import HTML, Markdown, display

# ADK Components
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.tools import googlesearch
from google.adk.tools import ToolContext
from google.adk.tools.agenttool import AgentTool
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part

print("✅ All libraries are ready to go!")

In [ ]:
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

In [ ]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-central1"        # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEX_AI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}
print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")

---
## Part 1: Your First Agent - The Research Assistant 🔬

Meet your first creation! The `research_agent` is a foundational assistant that can search the web and synthesize information on any topic.

```text
+--------------------------------------------------+
|            Research Assistant Agent 🔬            |
|            Model: gemini-2.5-flash               |
| Description: Researches topics and provides     |
| comprehensive summaries with cited sources       |
+--------------------------------------------------+
| 🔧 Tools:                                        |
| - Google Search                                  |
+--------------------------------------------------+
| 🧠 Capabilities:                                 |
| - Multi-source research  - Fact verification     |
| - Structured summaries   - Citation tracking     |
+--------------------------------------------------+
```

In [ ]:
def create_research_agent():
    """Create the Research Assistant agent"""
    return Agent(
        name="research_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in researching topics and synthesizing information from multiple sources.",
        instruction="""
        You are a Research Assistant 🔬 - a specialized AI that helps users understand complex topics.
        
        Your Mission:
        Transform research questions into comprehensive, well-cited summaries.

        Guidelines:
        1. Thorough Research: Use Google Search to find authoritative sources on the topic.
        2. Multiple Perspectives: Gather information from at least 2-3 different sources.
        3. Structured Output: Organize findings with clear headings and bullet points.
        4. Source Attribution: Always cite your sources with links.
        5. Critical Analysis: Note any conflicting information or areas of uncertainty.

        RETURN findings in MARKDOWN FORMAT with clear sections and citations.
        """,
        tools=[googlesearch]
    )

research_agent = create_research_agent()
print(f"🔬 Agent '{research_agent.name}' is created and ready for research!")

In [ ]:
async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, verbose: bool = True):
    """Initializes a runner and executes a query for a given agent and session."""
    if verbose:
        print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")
    
    runner = Runner(
        agent=agent,
        session_service=sessions_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if verbose:
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if verbose:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")

    return final_response

# Initialize Session Service globally
sessions_service = InMemorySessionService()
my_user_id = "agent_builder_001"

In [ ]:
async def run_research_test():
    research_session = await sessions_service.create_session(
        app_name=research_agent.name,
        user_id=my_user_id
    )
    query = "What are the latest developments in quantum computing for 2025-2026?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(research_agent, query, research_session, my_user_id)

# Execute the async function in the notebook
await run_research_test()

---
## Part 2: Custom Tools - Extending Agent Capabilities 🛠️

The real power of agents comes from connecting them to custom code logic and external application layers.

### 2.1 Function Tools: Calling External APIs
Let's create a functional tool asset that fetches real-time infrastructure data from the CoinGecko API.

In [ ]:
def get_crypto_price(cryptocurrency: str) -> dict:
    """Gets the current price and 24-hour change for a cryptocurrency.
    Args:
        cryptocurrency: The name or symbol of the cryptocurrency (e.g., "bitcoin", "ethereum").
    Returns:
        A dictionary containing price data and market information.
    """
    print(f"🛠️ TOOL CALLED: get_crypto_price(cryptocurrency='{cryptocurrency}')")

    cryptomap = {
        "bitcoin": "bitcoin", "btc": "bitcoin",
        "ethereum": "ethereum", "eth": "ethereum",
        "solana": "solana", "sol": "solana",
        "cardano": "cardano", "ada": "cardano",
        "dogecoin": "dogecoin", "doge": "dogecoin"
    }
    cryptoid = cryptomap.get(cryptocurrency.lower(), cryptocurrency.lower())

    try:
        url = f"https://api.coingecko.com/api/v3/coins/{cryptoid}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        return {
            "status": "success",
            "name": data["name"],
            "symbol": data["symbol"].upper(),
            "current_price_usd": f"${data['market_data']['current_price']['usd']:,.2f}",
            "price_change_24h": f"{data['market_data']['price_change_percentage_24h']:.2f}%",
            "market_cap_rank": data["market_cap_rank"],
            "last_updated": data["last_updated"]
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"Failed to fetch data: {e}"}

def get_stock_news(company: str) -> dict:
    """Gets recent news headlines about a company using web search.
    Args:
        company: The company name or stock ticker.
    Returns:
        A dictionary indicating news search should be performed.
    """
    print(f"🛠️ TOOL CALLED: get_stock_news(company='{company}')")
    return {
        "status": "success",
        "action": "search_required",
        "query": f"Latest {company} stock news today"
    }

In [ ]:
financial_agent = Agent(
    name="financial_analyst",
    model="gemini-2.5-flash",
    description="A financial analyst that provides market insights using real-time data.",
    instruction="""
    You are a Financial Analyst AI that helps users understand market conditions.
    Your Approach:
    1. When asked about cryptocurrencies, ALWAYS use the get_crypto_price tool first.
    2. For stock-related questions, use get_stock_news to identify what to search.
    3. Provide clear, actionable insights (not financial advice).
    4. Format numbers clearly and explain trends.

    Always clarify that you're providing market data information, not financial advice.
    """,
    tools=[get_crypto_price, get_stock_news, googlesearch]
)
print(f"📈 Agent '{financial_agent.name}' is ready to analyze markets!")

In [ ]:
async def run_financial_test():
    finance_session = await sessions_service.create_session(
        app_name=financial_agent.name,
        user_id=my_user_id
    )
    query = "What's the current price of Bitcoin and Solana? How have they performed in the last 24 hours?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(financial_agent, query, finance_session, my_user_id)

await run_financial_test()

---
### 2.2 Agent-as-a-Tool: Building Specialist Teams

The **Agent-as-a-Tool** pattern lets a single coordinating orchestrator agent treat downstream domain-specific agents as execution assets.

```text
+----------------------------------------------------------+
|               🎯 Project Manager Agent                   |
|  Orchestrates project planning by delegating to          |
|  specialist agents                                       |
+----------------------------------------------------------+
| 🔧 Tools:                                                |
| 1. call_technical_advisor                                |
| 2. call_risk_analyst                                     |
| 3. call_timeline_planner                                 |
+----------------------------------------------------------+
       /              |              \
      /               |               \
     ▼                ▼                ▼
+----------------+  +----------------+  +------------------+
| 💻 Technical   |  | ⚠️ Risk        |  | 📅 Timeline      |
|    Advisor     |  |    Analyst     |  |    Planner       |
+----------------+  +----------------+  +------------------+
| Evaluates      |  | Identifies     |  | Creates          |
| technical      |  | potential      |  | realistic        |
| feasibility    |  | risks &        |  | project          |
| & stack        |  | mitigations    |  | schedules        |
+----------------+  +----------------+  +------------------+
```

In [ ]:
technical_advisor = Agent(
    name="technical_advisor",
    model="gemini-2.5-flash",
    instruction="""
    You are a Senior Technical Advisor. When given a project idea:
    1. Evaluate technical feasibility.
    2. Recommend an appropriate modern, production-grade software technology stack.
    3. Identify key engineering bottlenecks and technical challenges.
    4. Estimate system architectural complexity (Low/Medium/High).
    Format output cleanly as markdown bullet points.
    """
)

risk_analyst = Agent(
    name="risk_analyst",
    model="gemini-2.5-flash",
    instruction="""
    You are a Risk Analyst. For any project:
    1. Identify top 3-5 potential deployment, operational, or safety risks.
    2. Rate each risk profile level (Low/Medium/High impact).
    3. Suggest clear software engineering mitigation strategies.
    """
)

timeline_planner = Agent(
    name="timeline_planner",
    model="gemini-2.5-flash",
    instruction="""
    You are a Project Timeline Specialist. Given engineering specifications:
    1. Break down workflow tasks into major sequential delivery phases.
    2. Estimate duration for each phase using weeks as metrics.
    3. Detail dependency milestones and assignable paths.
    """
)
print("✅ Specialist agents created: Technical Advisor, Risk Analyst, Timeline Planner")

In [ ]:
async def call_technical_advisor(project_description: str, tool_context: ToolContext) -> str:
    """Consults the Technical Advisor for engineering feasibility and stack choices."""
    print("--- TOOL CALL: call_technical_advisor ---")
    agent_tool = AgentTool(agent=technical_advisor)
    result = await agent_tool.run_async(
        args={"request": project_description},
        tool_context=tool_context
    )
    tool_context.state["technical_assessment"] = result
    return result

async def call_risk_analyst(project_description: str, tool_context: ToolContext) -> str:
    """Consults the Risk Analyst to evaluate project liabilities and failure vectors."""
    print("--- TOOL CALL: call_risk_analyst ---")
    tech_context = tool_context.state.get("technical_assessment", "")
    enhanced_request = f"Project: {project_description}\n\nTechnical Context: {tech_context}"

    agent_tool = AgentTool(agent=risk_analyst)
    result = await agent_tool.run_async(
        args={"request": enhanced_request},
        tool_context=tool_context
    )
    tool_context.state["risk_assessment"] = result
    return result

async def call_timeline_planner(project_description: str, tool_context: ToolContext) -> str:
    """Consults the Timeline Planner for scheduling construction sprints."""
    print("--- TOOL CALL: call_timeline_planner ---")
    full_context = f"""
    Project: {project_description}
    Technical Assessment: {tool_context.state.get("technical_assessment", "Not yet assessed")}
    Risk Assessment: {tool_context.state.get("risk_assessment", "Not yet assessed")}
    """
    agent_tool = AgentTool(agent=timeline_planner)
    return await agent_tool.run_async(
        args={"request": full_context},
        tool_context=tool_context
    )

In [ ]:
project_manager_agent = Agent(
    name="project_manager",
    model="gemini-2.5-flash",
    description="Orchestrates complete architectural planning by routing to specialist tools.",
    tools=[call_technical_advisor, call_risk_analyst, call_timeline_planner],
    instruction="""
    You are an AI Technical Project Manager. Your job is to compile structural design reports.
    
    Execution Run Matrix Pipeline:
    1. Exec call_technical_advisor to determine stack composition parameters.
    2. Exec call_risk_analyst to catch operational issues.
    3. Exec call_timeline_planner to outline development velocity metrics.

    Synthesize into a master roadmap including Executive Summaries, Tech stacks, Risk controls, and Gantt milestones.
    """
)
print(f"✅ Orchestrator Agent '{project_manager_agent.name}' is online and operational!")

In [ ]:
async def run_project_planning():
    pm_session = await sessions_service.create_session(
        app_name=project_manager_agent.name,
        user_id=my_user_id
    )
    query = """
    I want to build a mobile app that uses AI to identify local plants from user photos 
    and provides contextual automated irrigation schedules. Target platforms: iOS and Android.
    """
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(project_manager_agent, query, pm_session, my_user_id)

await run_project_planning()

---
## Part 3: Conversational Memory - The Adaptive Tutor 📚

Memory transforms disconnected requests into true context-aware system workflows. Let's design an agent that tracks thread tracking arrays across multi-turn runs.

```text
+-----------------------------------------------------+
|             Adaptive Learning Tutor 📚              |
|             Model: gemini-2.5-flash                 |
| Description: Teaches concepts progressively, adapts  |
| to student understanding, remembers progress         |
+-----------------------------------------------------+
| 🔧 Tools:                                           |
| - Google Search (for examples/references)           |
+-----------------------------------------------------+
| 🧠 Capabilities:                                    |
| - Tracks learning progress across conversation turns|
| - Adjusts explanation complexity                    |
+-----------------------------------------------------+
```

In [ ]:
def create_tutor_agent():
    return Agent(
        name="adaptive_tutor",
        model="gemini-2.5-flash",
        description="An adaptive learning tutor that adjusts teaching based on student profile tracking.",
        instruction="""
        You are an Adaptive Learning Tutor 📚.
        
        System Rules:
        1. Access context logs continuously to reference historical student assertions.
        2. Adjust depth metrics dynamically if confusion conditions surface.
        3. Finish segments with explicit content confirmation testing questions.
        """,
        tools=[googlesearch]
    )

tutor_agent = create_tutor_agent()
print(f"📚 Agent '{tutor_agent.name}' initialized.")

---
### Scenario 3a: Memory in Action (Single Continuous Session Execution) ✅

In [ ]:
async def run_tutor_with_memory():
    print("### 🧠 DEMO: ADAPTIVE TUTOR WITH PERSISTENT SESSION MEMORY ###\n")
    learning_session = await sessions_service.create_session(
        app_name=tutor_agent.name,
        user_id=my_user_id
    )
    print(f"📖 Active Session Pipeline Handle: {learning_session.id}\n")

    # Turn 1
    q1 = "Hi! I want to learn about machine learning. I'm a visual learner and work best with concrete engineering examples. Background is CS."
    await run_agent_query(tutor_agent, q1, learning_session, my_user_id, verbose=False)

    # Turn 2
    q2 = "Awesome. Can you breakdown supervised learning mechanics using a functional paradigm case?"
    await run_agent_query(tutor_agent, q2, learning_session, my_user_id, verbose=False)

    # Turn 3
    q3 = "Hold on, clarify structural distribution delta classification vs regression models for me?"
    await run_agent_query(tutor_agent, q3, learning_session, my_user_id, verbose=True)

await run_tutor_with_memory()

---
### Scenario 3b: Memory Anti-Pattern Demonstration (New Session Each Turn) ❌

Watch what happens when you accidentally break session state continuity.

In [ ]:
async def run_tutor_without_memory():
    print("\n" + "*"*60)
    print("### ❌ ANTI-PATTERN: NEW ISOLATED SESSION INSTANTIATED EACH TURN ###")
    print("*"*60 + "\n")
    
    # Turn 1
    session_1 = await sessions_service.create_session(app_name=tutor_agent.name, user_id=my_user_id)
    q1 = "Hi! I want to learn about deep neural networks. I am completely new to linear algebra."
    print(f"Session 1 ID: {session_1.id}")
    await run_agent_query(tutor_agent, q1, session_1, my_user_id, verbose=False)

    # Turn 2: STATE IS LOST!
    session_2 = await sessions_service.create_session(app_name=tutor_agent.name, user_id=my_user_id)
    q2 = "Based on my math background that I just mentioned, where should I start?"
    print(f"\nSession 2 ID (NEW ISOLATED INSTANCE): {session_2.id}")
    await run_agent_query(tutor_agent, q2, session_2, my_user_id, verbose=True)

await run_tutor_without_memory()

---
## Part 4: Putting It All Together - The Knowledge Navigator 🧭

Let's assemble a production-grade multi-agent intent router topology.

```text
+--------------------------------------------------------------+
|                    🧭 Knowledge Navigator                     |
| Top-level orchestrator that routes queries to specialists    |
+--------------------------------------------------------------+
| 🔧 Tools:                                                    |
| - research_specialist (Agent-as-Tool)                        |
| - code_specialist     (Agent-as-Tool)                        |
| - creative_specialist (Agent-as-Tool)                        |
+--------------------------------------------------------------+
      /                    |                    \
     /                     |                     \
    ▼                      ▼                      ▼
+------------------+   +------------------+   +------------------+
| 🔬 Research      |   | 💻 Code          |   | 🎨 Creative      |
|    Specialist    |   |    Specialist    |   |    Specialist    |
+------------------+   +------------------+   +------------------+
| Deep research    |   | Code generation  |   | Writing &        |
| & fact-finding   |   | & debugging      |   | brainstorming    |
+------------------+   +------------------+   +------------------+
```

In [ ]:
research_specialist = Agent(
    name="research_specialist",
    model="gemini-2.5-flash",
    instruction="Execute complete domain analytics data verification sweeps.",
    tools=[googlesearch]
)

code_specialist = Agent(
    name="code_specialist",
    model="gemini-2.5-flash",
    instruction="Write enterprise-ready, tested code profiles containing error handling modules."
)

creative_specialist = Agent(
    name="creative_specialist",
    model="gemini-2.5-flash",
    instruction="Generate unique, out-of-the-box conceptual themes and branding frameworks."
)
print("✅ Core Specialists Cluster initialized successfully.")

In [ ]:
async def consult_research_specialist(query: str, tool_context: ToolContext) -> str:
    """Delegates deep analytical and factual synthesis tasks to the Research Specialist."""
    print("--- 🔀 ROUTING VECTOR -> Research Specialist ---")
    return await AgentTool(agent=research_specialist).run_async(args={"request": query}, tool_context=tool_context)

async def consult_code_specialist(query: str, tool_context: ToolContext) -> str:
    """Delegates engineering, syntax construction, and refactoring scripts to the Code Specialist."""
    print("--- 🔀 ROUTING VECTOR -> Code Specialist ---")
    return await AgentTool(agent=code_specialist).run_async(args={"request": query}, tool_context=tool_context)

async def consult_creative_specialist(query: str, tool_context: ToolContext) -> str:
    """Delegates branding identity, content production, and ideation to the Creative Specialist."""
    print("--- 🔀 ROUTING VECTOR -> Creative Specialist ---")
    return await AgentTool(agent=creative_specialist).run_async(args={"request": query}, tool_context=tool_context)

In [ ]:
navigator_agent = Agent(
    name="knowledge_navigator",
    model="gemini-2.5-flash",
    description="Intelligent system runtime router connecting specialized functional sub-nodes.",
    tools=[consult_research_specialist, consult_code_specialist, consult_creative_specialist],
    instruction="""
    You are the Knowledge Navigator 🧭 - an intelligent structural context router.
    
    Routing Matrix Rules:
    - Research vectors -> consult_research_specialist
    - Engineering/Code arrays -> consult_code_specialist  
    - Ideation/Copy structural creation -> consult_creative_specialist

    For composite prompts, call multiple specialized subsystems sequentially and merge the outputs into a coherent technical layout.
    """
)
print(f"🧭 Navigator Agent Node '{navigator_agent.name}' online.")

In [ ]:
async def run_navigator_demo():
    nav_session = await sessions_service.create_session(
        app_name=navigator_agent.name,
        user_id=my_user_id
    )
    
    # Composite challenge prompt requiring execution inside multiple sub-nodes
    complex_query = """
    I'm designing an open-source weather ingestion infrastructure. Can you:
    1. Summarize how historical climatic data formats like NetCDF4 map into standard arrays (Research).
    2. Write a clean Python function using xarray to read an engineering data sample (Code).
    3. Brainstorm a cool name for this open-source package (Creative).
    """
    print(f"🗣️ User Composite Request:\n{complex_query}\n")
    await run_agent_query(navigator_agent, complex_query, nav_session, my_user_id)

await run_navigator_demo()

---
## 🎉 Congratulations! 🎉

You have successfully completed your journey from single-agent runtimes to building an operational multi-agent network!

### Key Architectural Milestones Mastered:

| Module Concept | Practical Application System Asset |
| :--- | :--- |
| **Foundational Agents** | Configured runtime instructions, specified LLM backends, and connected system tools. |
| **Custom Function Tools** | Connected external REST APIs directly to agent runtime decisions. |
| **Agent-as-a-Tool Patterns** | Wrapped sub-agents into operational tools for high-level orchestrators. |
| **Session Management State** | Tracked conversational context and prevented session state leakage. |
| **Multi-Agent Orchestration** | Designed a declarative routing matrix for parallel task execution. |

```text
  //\         //\        //\
o-''|/).  ( o.o )       ( -.- )      ( ^^ )     o-''|/).    ( ^^ )
/|)     )    > ^ <         > * <        >🏆<         /|)     )     / >🌟< 
\    /                                              \    /         /   
(/ (/                                               (/ (/        (|_)
```
**You are now fully prepared to build production-grade AI Agent systems at a national scale! 🚀**